[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/spin_transformations/spin_transformations.ipynb)

# Spin Transformations

This example computes spin transformations: bendings of a surface that keep every angle. A quaternion at each vertex, a turn and a scale, acts on the edges around it, and [Crane, Pinkall and Schröder (2011)](https://www.cs.cmu.edu/~kmcrane/Projects/SpinTransformations/paper.pdf) choose the quaternions so that the surface's mean curvature changes by a prescribed amount. Repeated, the transformation becomes a conformal curvature flow that rounds Spot the cow into a sphere; asked for a constant change, its fields are the spinor spherical harmonics, and they spin the sphere into Dirac spheres.

In the algebra of three-dimensional space the quaternions are the even multivectors, and the vertices and edges are points, antivectors, intrinsic like all geometry, which in three dimensions are bivectors. Every operator is a sparse extensor, a linear map between fields on the vertices, edges and faces of a mesh in which each element couples to only a few others by a multivector: the Dirac operator couples each face to its corners by bivectors, so it takes quaternion fields to quaternion fields, and the energy of a quaternion field, its area-weighted squared residual, `~residual * face_areas * residual`, couples vertices by quaternions. The new edges come from sandwiching each old edge with the quaternions along it, `quaternions << edges`.

In the quaternionic notation of Crane, Pinkall and Schröder the quaternions are the solution $\lambda$ of the eigenvalue problem $(D - \rho)\,\lambda = \gamma\lambda$ of least $|\gamma|$, and each new edge is $\tilde e = \bar\lambda\, e\, \lambda$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor, spdiag
from examples import instantiate
from examples.util.animation import save_figure, show_animation
from examples.surfaces.spin_transformations import render
from examples.surfaces.spin_transformations.core import spot

# Meshes, their incidences as sparse maps between fields, and their areas and normals, in the algebra of space.
mesh_module = instantiate("examples.util.mesh", ga)
Mesh, as_ga_sparse, as_scalar = mesh_module.Mesh, mesh_module.as_ga_sparse, mesh_module.as_scalar
mv = mesh_module.ctx.multivector
Scalar = ga.gatype.scalar()
# Vertices and edges are points, antivectors, intrinsic like all geometry; quaternions are the even multivectors.
Point = ga.gatype.antivector()
Vector = ga.gatype.vector()
Even = ga.gatype.even()

np.set_printoptions(precision=4, suppress=True)

# A unit sphere, its icosahedron split so many times.
n_subdivisions = 3
mesh = Mesh.icosphere(n_subdivisions)
print(mesh.vertices.gatype, mesh.edges.shape, mesh.faces.shape)

## 1. Operators between fields

A field is an extensor with one element per vertex, edge or face, such as `mesh.vertices`, `Point[V]`. An operator between fields is a sparse extensor, a linear map that couples each element of one field to a few elements of another, each coupling an extensor. Here the couplings are multivectors, nullary extensors, which act by their product: times a field, each coupling multiplies its input element into its output element, and the products are summed; times another operator it composes the two; and `~` is its reverse, with inputs and outputs swapped and every coupling reversed. The mesh's incidences say which elements couple. These operators couple through scalars.

In [ ]:
def triangle_edges(mesh: Mesh) -> Point:
    """`[3] Point[F]`: the edge facing each corner of each face, from the corner after it to the corner after that."""
    corners = mesh.corners                                                     # [3] Point[F]
    return corners[[2, 0, 1]] - corners[[1, 2, 0]]


def triangle_areas(mesh: Mesh) -> Scalar:
    """`Scalar[F]`: half the size of each face's plane, the commutator of two of its edges."""
    edges = triangle_edges(mesh)                                               # [3] Point[F]
    return edges[1].commutator(edges[0]).norm() / 2


def dual_areas(mesh: Mesh) -> Scalar:
    """`Scalar[V]`: a third of the area of each face at each vertex."""
    thirds = as_ga_sparse(mesh.faces, as_scalar(np.ones_like(mesh.faces) / 3))   # [F, V] Scalar
    return ~thirds * triangle_areas(mesh)


def cotangent_weights(mesh: Mesh) -> Scalar:
    """`Scalar[E]`: half the cotangents of the angles facing each edge. The cotangent at a corner is one edge's slice
    read along the other, over the size of the plane they span; head to tail, the corner's own edges point the other
    way."""
    edges = triangle_edges(mesh)                                               # [3] Point[F]
    after, before = edges[[1, 2, 0]], edges[[2, 0, 1]]                         # [3] Point[F] each
    cotangents = -(after.dual_inverse() & before) / after.commutator(before).norm()   # [3] Scalar[F]
    return ~mesh.by_corner(cotangents / 2) * as_scalar(np.ones(len(mesh.faces))).field()


def face_normals(mesh: Mesh) -> Vector:
    """`Vector[F]`: each face's unit normal, outwards: where two of its edges meet, over twice its area."""
    edges = triangle_edges(mesh)                                               # [3] Point[F]
    return (edges[0] & edges[1]) / (2 * triangle_areas(mesh))


def vertex_normals(mesh: Mesh) -> Vector:
    """`Vector[V]`: the sum of the faces' normals at each vertex, each twice its face's area long, made unit."""
    edges = triangle_edges(mesh)                                               # [3] Point[F]
    spread = as_ga_sparse(mesh.faces, as_scalar(np.ones_like(mesh.faces, dtype=float)))   # [F, V] Scalar
    return (~spread * (edges[0] & edges[1])).normalized()


def unit_volume(mesh: Mesh) -> Scalar:
    """`Scalar`: the sense the counter-clockwise faces give space, plus or minus one: the faces' corners meeting
    about the origin, summed over the solid, over its size."""
    corners = mesh.corners                                                     # [3] Point[F]
    volume = (corners[0] & corners[1] & corners[2]).sites.sum()                # [] Scalar
    return volume / volume.norm()

In [ ]:
# the face-vertex and edge-vertex incidences, and the edges' orientations
face_corners, edge_ends = mesh.faces, mesh.edges                        # [F, 3], [E, 2]
orientations = np.ones_like(edge_ends) * [-1, 1]                        # [E, 2]

# the boundary, taking each edge's tail from its head, and the means over each edge and each face
boundary = as_ga_sparse(edge_ends, as_scalar(orientations))             # [E, V] Scalar
edge_means = as_ga_sparse(edge_ends, as_scalar(np.ones_like(edge_ends) / 2))   # [E, V] Scalar
face_means = as_ga_sparse(face_corners, as_scalar(np.ones_like(face_corners) / 3))   # [F, V] Scalar

# diagonal operators: the triangle areas, the vertex areas, and each edge's cotangent weight
face_areas = spdiag(triangle_areas(mesh))                               # [F, F] Scalar
inverse_face_areas = spdiag(1 / triangle_areas(mesh))                   # [F, F] Scalar
vertex_areas = spdiag(dual_areas(mesh))                                 # [V, V] Scalar
edge_weights = spdiag(cotangent_weights(mesh))                          # [E, E] Scalar

edges = boundary * mesh.vertices                                        # Point[E]

In [ ]:
print(boundary.gatype, boundary.shape)
print(edges.gatype)

## 2. A change in mean curvature

The transformation is asked for by a change in each face's mean curvature, `curvature_change`, a scalar field on the faces. Here it grows along y, each face's centroid paired with the direction y, `centroids & mv.y`: the surface is to bend more on one side and less on the other, red where it is to bend more.

In [ ]:
# How strongly the mean curvature changes along y.
dipole = 0.5

In [ ]:
centroids = face_means * mesh.vertices                                  # Point[F]
curvature_change = (centroids & mv.y) * dipole                          # Scalar[F]

save_figure(render.draw_surface(mesh, curvature_change), "spin_transformations_curvature");

## 3. The Dirac operator

The Dirac operator sends a quaternion field on the vertices to a field on the faces. Crane, Pinkall and Schröder build it edge by edge: each face takes the quaternion at each corner by the edge that corner faces, over twice the face's area, so the cells of `dirac` are bivectors and it takes a quaternion field to a quaternion field on the faces. The same operator is the geometric derivative, with each face's unit normal multiplied in on the right. On each face a field on the vertices is linear in between, and `derivative` takes it to its gradient there: the coupling to each corner is the gradient of that corner's hat function, the edge the corner faces paired with the face's unit normal, `edge | normal`, the edge turned a quarter turn within the face, over twice the face's area. `derivative * field` is the field's geometric derivative, and with the normal on the right it is `dirac * field`. The `potential` puts the curvature change, read against the sense the surface's counter-clockwise faces give space, `mesh.unit_volume`, on each face's mean. The transformation's quaternions solve `dirac * field == potential * field`, so `residual = dirac - potential`, its cells even, takes a field to how far it is from a transformation. The energy of a field is its area-weighted squared residual, `~residual * face_areas * residual`, its cells quaternions too.

In the quaternionic matrix notation of Crane, Pinkall and Schröder this reads as $D_{ij} = -\tfrac{1}{2\mathcal{A}_i} e_j$, with $e_j$ the edge as an imaginary quaternion, minus its bivector in Hamilton's convention, $A = D - R$, and the energy $A^{H} M_F A$.

In [ ]:
def dirac_operator(mesh: Mesh) -> SparseExtensor:
    """`[F, V] Point`: each face takes each corner by the edge it faces, over twice the face's area."""
    return spdiag(1 / triangle_areas(mesh)) * mesh.at_corners(triangle_edges(mesh)) * 0.5


def geometric_derivative(mesh: Mesh) -> SparseExtensor:
    """`[F, V] Vector`: on each face the gradient of each corner's hat function, the edge facing it paired with the
    face's unit normal, over twice the face's area."""
    return spdiag(1 / triangle_areas(mesh)) * mesh.at_corners(triangle_edges(mesh) | face_normals(mesh)) * 0.5


def energy(mesh: Mesh, curvature_change: Scalar) -> SparseExtensor:
    """`[V, V] Even`: the area-weighted squared residual of `dirac * field == potential * field`, for a change in
    each face's mean curvature `Scalar[F]`."""
    face_means = as_ga_sparse(mesh.faces, as_scalar(np.ones_like(mesh.faces) / 3))   # [F, V] Scalar
    # The curvature change, read against the sense the faces give space, on each face's mean.
    potential = spdiag(curvature_change * unit_volume(mesh)) * face_means       # [F, V] Scalar
    residual = dirac_operator(mesh) - potential                                # [F, V] Even
    return ~residual * spdiag(triangle_areas(mesh)) * residual


dirac, derivative = dirac_operator(mesh), geometric_derivative(mesh)            # [F, V] Point, [F, V] Vector
sphere_energy = energy(mesh, curvature_change)                                  # [V, V] Even

In [ ]:
print(derivative.gatype, dirac.gatype)
print(sphere_energy.gatype, sphere_energy.shape)

## 4. The quaternion field

The quaternions are the field of least energy per unit of vertex area, the least eigenpair of `energy` against the vertex areas. An eigensolve has to know what a linear map acts on, so `energy * Even` and `vertex_areas * Even` turn its multivectors into maps on quaternions. The field is fixed only up to a constant factor, which would turn and scale the whole surface, so it is divided by its area-weighted mean. The colours show how much each face's mean quaternion scales it, red where it grows and blue where it shrinks.

In [ ]:
def least_field(mesh: Mesh, curvature_change: Scalar) -> Even:
    """The quaternion field `Even[V]` of least energy per unit of vertex area, divided by its area-weighted mean."""
    vertex_areas = spdiag(dual_areas(mesh))                                   # [V, V] Scalar
    _, modes = (energy(mesh, curvature_change) * Even).eigh(vertex_areas * Even, 1)   # [1] Even[V]
    mean = (vertex_areas * modes[0]).sites.sum() / dual_areas(mesh).sites.sum()   # [] Even
    return modes[0] / mean


quaternions = least_field(mesh, curvature_change)                               # Even[V]
scaling = (face_means * quaternions).scalar_norm_squared() - 1                  # Scalar[F]

save_figure(render.draw_surface(mesh, scaling), "spin_transformations_scaling");

## 5. The new surface

Each edge is turned and scaled by the field along it, the reverse sandwich `quaternion << edge` with the quaternion running linearly from one end to the other, integrated by Simpson's rule: the turns at the two ends and twice the turn at the middle. The turns at the ends are averaged as maps, `quaternions << Point`, the turn at each vertex as a map on points, so that `edge_means` carries them to the edges before they meet the edge they turn; the turn at the middle is the sandwich of the mean quaternion. The new vertices are those whose edges match the turned ones best, weighted by the cotangent Laplacian: the least-squares solution of the Laplacian acting on points, `laplacian * Point`, which leaves the surface centred. The checkerboard is the sphere's: a conformal map keeps its squares square.

In the quaternionic notation of Crane, Pinkall and Schröder this reads as $\tilde e_{ij} = \tfrac13 \bar\lambda_i e_{ij} \lambda_i + \tfrac16 \bar\lambda_i e_{ij} \lambda_j + \tfrac16 \bar\lambda_j e_{ij} \lambda_i + \tfrac13 \bar\lambda_j e_{ij} \lambda_j$.

In [ ]:
def turned(mesh: Mesh, quaternions: Even) -> Point:
    """The vertices `[...] Point[V]` whose edges best match the mesh's edges turned and scaled by the quaternion field
    `[...] Even[V]` along them, matched by the cotangent Laplacian, centred."""
    boundary = as_ga_sparse(mesh.edges, as_scalar(np.ones_like(mesh.edges) * [-1, 1]))   # [E, V] Scalar
    edge_means = as_ga_sparse(mesh.edges, as_scalar(np.ones_like(mesh.edges) / 2))     # [E, V] Scalar
    edge_weights = spdiag(cotangent_weights(mesh))                                     # [E, E] Scalar
    edges = boundary * mesh.vertices                                           # Point[E]
    # Simpson's rule along each edge: the turns at its two ends, averaged as maps, and twice the turn at its middle.
    moved = ((edge_means * (quaternions << Point))(edges) + 2 * ((edge_means * quaternions) << edges)) / 3   # [...] Point[E]
    return ((~boundary * edge_weights * boundary) * Point).lstsq(~boundary * edge_weights * moved)


deformed = mesh.copy(vertices=turned(mesh, quaternions))

save_figure(render.draw_textured(deformed, mesh), "spin_transformations_deformed");

## 6. Conformal curvature flow

Repeating the transformation, each time asking for a share of every face's departure from the mean curvature to be taken away, rounds a surface step by step while every step keeps its angles. Spot the cow, by Keenan Crane, draws in her legs, horns and ears and rounds into a sphere; a checkerboard laid on the cow keeps its squares square all the way, since no step changes an angle.

In [ ]:
# The share of each face's curvature departure each step removes, the steps, and the duration of a frame.
rate = 0.2
n_flow_steps = 15
duration_ms = 160

In [ ]:
def mean_curvature(mesh: Mesh) -> Scalar:
    """Each face's signed mean curvature `Scalar[F]`: the cotangent Laplacian of the positions is the area-integrated
    mean-curvature normal at each vertex, met with the vertex's normal and read against the sense the faces give space,
    over twice the vertex's area, and averaged onto the faces."""
    boundary = as_ga_sparse(mesh.edges, as_scalar(np.ones_like(mesh.edges) * [-1, 1]))   # [E, V] Scalar
    face_means = as_ga_sparse(mesh.faces, as_scalar(np.ones_like(mesh.faces) / 3))      # [F, V] Scalar
    curving = (~boundary * spdiag(cotangent_weights(mesh)) * boundary) * mesh.vertices   # Point[V]
    integrated = (curving & vertex_normals(mesh)) * unit_volume(mesh)            # Scalar[V]
    return face_means * (integrated / (2 * dual_areas(mesh)))


def recentred(mesh: Mesh) -> Mesh:
    """The mesh moved to its centre and scaled to unit mean distance from it: what a spin transformation leaves free."""
    vertices = mesh.vertices - mesh.vertices.sites.mean()
    return mesh.copy(vertices=vertices / vertices.norm().sites.mean())


def rounding(mesh: Mesh, n_steps: int, rate: float):
    """The mesh at the start and after each step that takes away a share `rate` of every face's departure from the
    area-weighted mean curvature, a change in curvature, by one spin transformation."""
    mesh = recentred(mesh)
    yield mesh
    for _ in range(n_steps):
        curvature = mean_curvature(mesh)                                       # Scalar[F]
        mean = (curvature * triangle_areas(mesh)).sites.sum() / triangle_areas(mesh).sites.sum()   # [] Scalar
        mesh = recentred(mesh.copy(vertices=turned(mesh, least_field(mesh, -rate * (curvature - mean)))))
        yield mesh


show_animation(render.animate_rounding(rounding(spot(), n_flow_steps, rate)), "spin_rounding", duration_ms)

## 7. Dirac spheres

With `rho` a constant `n`, the fields of zero energy are the eigenfields of the Dirac operator itself. Their eigenvalues are the integers except −1, each with multiplicity one more than itself, and spinning the sphere by one gives a Dirac sphere. `dirac_spheres` runs the energy above with that constant and spins the sphere by each of the fields of least energy, one row per eigenvalue. The higher states are finer, so the sphere is split once more.

Crane, Pinkall and Schröder name the physics these fields come from (§6.2):

> Eigenfunctions of D on the unit sphere are called the spinor spherical harmonics, which can be used to compute fundamental solutions to the Dirac equation for the configuration of a spin-1/2 particle such as an electron in a spherically symmetric potential.

In [ ]:
# The finer sphere's subdivisions, the eigenvalues shown, and the states shown of each.
n_dirac_subdivisions = 4
eigenvalues = (1, 2, 3, -2, -3)
n_states = 3

In [ ]:
def dirac_spheres(mesh: Mesh, eigenvalue: int, n_states: int) -> tuple[Scalar, list[Mesh]]:
    """The energies of the `n_states` fields of least energy for a constant curvature change, the eigenvalue, and the
    surfaces they spin the mesh into; the fields at unit size per unit of vertex area, as they come."""
    constant = as_scalar(np.full(len(mesh.faces), float(eigenvalue))).field()   # Scalar[F]
    energies, fields = (energy(mesh, constant) * Even).eigh(spdiag(dual_areas(mesh)) * Even, n_states)   # [states] Scalar, [states] Even[V]
    return energies, [mesh.copy(vertices=vertices) for vertices in turned(mesh, fields)]


fine = Mesh.icosphere(n_dirac_subdivisions)
spun = {n: dirac_spheres(fine, n, n_states) for n in eigenvalues}               # eigenvalue: ([states] Scalar, [states] Mesh)

save_figure(render.draw_dirac({n: spheres for n, (_, spheres) in spun.items()}, fine), "dirac_spheres");

In [ ]:
# checks
# Crane's Dirac operator is the geometric derivative with each face's unit normal on the right, where it
# anticommutes with the in-plane gradients; and the geometric derivative of position is twice the volume
# element on every face: the dimension of the surface, with no curl.
np.testing.assert_allclose((spdiag(mesh.face_normals) * derivative + dirac).cells.kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((derivative * mesh.vertices).kernel, np.broadcast_to([0, 0, 0, 2], (len(mesh.faces), 4)), atol=1e-12)
# Every corner keeps its angle, up to the mesh's resolution.
np.testing.assert_allclose((deformed.corner_cosines() - mesh.corner_cosines()).to_array(), 0.0, atol=0.05)
# The Dirac spheres' fields have zero energy, up to the mesh's resolution against the spectrum's unit spacing.
np.testing.assert_allclose(np.stack([energies.to_array().ravel() for energies, _ in spun.values()]), 0.0, atol=0.05)